# BioMistral 7B DARE Fine-Tuning on Triples (Colab)

This notebook fine-tunes dizza01/BioMistral-7B-DARE with QLoRA on triples SFT data in Google Drive.

Expected files in Drive:
- /content/drive/MyDrive/triples/train_dev_val/sft_train.jsonl
- /content/drive/MyDrive/triples/train_dev_val/sft_dev.jsonl
- /content/drive/MyDrive/triples/train_dev_val/sft_test.jsonl

Outputs are pushed to separate repos so the baseline deployment repo stays unchanged.

In [ ]:
!pip -q install -U transformers datasets peft accelerate bitsandbytes trl huggingface_hub sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged into Hugging Face as:', whoami()['name'])
else:
    print('HF_TOKEN is missing in Colab secrets. Add it, then rerun this cell.')

In [ ]:
import os
import random
from dataclasses import dataclass

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from huggingface_hub import HfApi

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
@dataclass
class Config:
    base_model: str = 'dizza01/BioMistral-7B-DARE'

    train_jsonl: str = '/content/drive/MyDrive/triples/train_dev_val/sft_train.jsonl'
    dev_jsonl: str = '/content/drive/MyDrive/triples/train_dev_val/sft_dev.jsonl'
    test_jsonl: str = '/content/drive/MyDrive/triples/train_dev_val/sft_test.jsonl'

    adapter_output_dir: str = '/content/drive/MyDrive/triples/train_dev_val/biomistral7b_dare_triples_lora'
    merged_output_dir: str = '/content/drive/MyDrive/triples/train_dev_val/biomistral7b_dare_triples_merged'

    adapter_repo_id: str = 'dizza01/biomistral-7b-dare-triples-lora'
    merged_repo_id: str = 'dizza01/biomistral-7b-dare-triples-lora-merged'
    push_adapter: bool = True
    push_merged: bool = True

    max_seq_len: int = 2048
    epochs: float = 3.0
    learning_rate: float = 2e-4
    train_batch_size: int = 1
    eval_batch_size: int = 1
    grad_accum_steps: int = 16
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0

    lora_r: int = 64
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    save_steps: int = 25
    eval_steps: int = 25
    logging_steps: int = 5

    use_4bit: bool = True

cfg = Config()
cfg

In [ ]:
data_files = {'train': cfg.train_jsonl, 'validation': cfg.dev_jsonl}
raw = load_dataset('json', data_files=data_files)

def to_text(row):
    text = str(row.get('text', '')).strip()
    if text:
        return {'text': text}

    messages = row.get('messages', [])
    parts = []
    for m in messages:
        role = m.get('role', 'user')
        content = m.get('content', '')
        parts.append(f'<|im_start|>{role}\n{content}<|im_end|>')
    return {'text': '\n'.join(parts)}

train_text = raw['train'].map(to_text, remove_columns=raw['train'].column_names)
eval_text = raw['validation'].map(to_text, remove_columns=raw['validation'].column_names)

print('Train rows:', len(train_text))
print('Eval rows:', len(eval_text))
print(train_text[0]['text'][:700])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_batch(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=cfg.max_seq_len,
    )
    out['labels'] = out['input_ids'].copy()
    return out

train_tok = train_text.map(tokenize_batch, batched=True)
eval_tok = eval_text.map(tokenize_batch, batched=True)

keep = {'input_ids', 'attention_mask', 'labels'}
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in keep])
eval_tok = eval_tok.remove_columns([c for c in eval_tok.column_names if c not in keep])

train_tok.set_format(type='torch')
eval_tok.set_format(type='torch')

print(train_tok[0].keys())

In [ ]:
if cfg.use_4bit and not torch.cuda.is_available():
    raise RuntimeError('use_4bit=True requires CUDA.')

model_kwargs = {'trust_remote_code': True}
if cfg.use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    model_kwargs['quantization_config'] = bnb_config
    model_kwargs['device_map'] = 'auto'
else:
    model_kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(cfg.base_model, **model_kwargs)
model.config.use_cache = False

if cfg.use_4bit:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

common_args = dict(
    output_dir=cfg.adapter_output_dir,
    num_train_epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    eval_steps=cfg.eval_steps,
    save_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    lr_scheduler_type='cosine',
    bf16=bf16,
    fp16=fp16,
    gradient_checkpointing=True,
    report_to='none',
    dataloader_pin_memory=False,
)

try:
    training_args = TrainingArguments(
        **common_args,
        eval_strategy='steps',
    )
except TypeError:
    training_args = TrainingArguments(
        **common_args,
        evaluation_strategy='steps',
    )

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
)

trainer.train()

In [ ]:
trainer.save_model(cfg.adapter_output_dir)
tokenizer.save_pretrained(cfg.adapter_output_dir)
print('Saved adapter and tokenizer to', cfg.adapter_output_dir)

In [ ]:
if cfg.push_adapter:
    api = HfApi()
    api.create_repo(repo_id=cfg.adapter_repo_id, repo_type='model', exist_ok=True)
    trainer.model.push_to_hub(cfg.adapter_repo_id)
    tokenizer.push_to_hub(cfg.adapter_repo_id)
    print('Adapter pushed to https://huggingface.co/' + cfg.adapter_repo_id)
else:
    print('Skipping adapter push. Set cfg.push_adapter=True to upload.')

In [ ]:
import textwrap
from pathlib import Path

os.makedirs(cfg.merged_output_dir, exist_ok=True)

print('Loading base model for merge...')
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print('Loading trained adapter...')
peft_model = PeftModel.from_pretrained(base_model, cfg.adapter_output_dir)

print('Merging adapter into base model...')
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(cfg.merged_output_dir, safe_serialization=True, max_shard_size='5GB')
tokenizer.save_pretrained(cfg.merged_output_dir)

requirements_text = '\n'.join([
    'transformers>=4.51.3',
    'torch>=2.2.0',
    'accelerate>=0.34.0',
    'safetensors>=0.4.0',
    'sentencepiece>=0.2.0',
]) + '\n'

handler_text = textwrap.dedent('''
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class EndpointHandler:
    def __init__(self, path: str = ''):
        model_dir = path or '/repository'

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            trust_remote_code=True,
        )

        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_dir,
            trust_remote_code=True,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map='auto',
        )
        self.model.eval()

    def __call__(self, data):
        inputs = data.get('inputs', '')
        params = data.get('parameters', {}) or {}

        max_new_tokens = int(params.get('max_new_tokens', 128))
        temperature = float(params.get('temperature', 0.0))
        top_p = float(params.get('top_p', 1.0))
        do_sample = bool(params.get('do_sample', temperature > 0))

        if isinstance(inputs, list):
            prompt = self.tokenizer.apply_chat_template(
                inputs,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = str(inputs)

        enc = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=do_sample,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_ids = out[0][enc['input_ids'].shape[-1]:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        return {'generated_text': text}
''').strip() + '\n'

Path(cfg.merged_output_dir, 'requirements.txt').write_text(requirements_text, encoding='utf-8')
Path(cfg.merged_output_dir, 'handler.py').write_text(handler_text, encoding='utf-8')

print('Merged endpoint package saved to:', cfg.merged_output_dir)
print('Included files: model shards + tokenizer + requirements.txt + handler.py')

In [ ]:
if cfg.push_merged:
    api = HfApi()
    api.create_repo(repo_id=cfg.merged_repo_id, repo_type='model', exist_ok=True)
    api.upload_folder(
        repo_id=cfg.merged_repo_id,
        repo_type='model',
        folder_path=cfg.merged_output_dir,
    )
    print('Merged endpoint-ready model pushed to https://huggingface.co/' + cfg.merged_repo_id)
else:
    print('Skipping merged push. Set cfg.push_merged=True to upload.')

## Suggested evaluation command

Use your new endpoint URL after deployment:

../../.venv/bin/python eval/run_faithfulness_eval_updated.py \
    --triples eval/evaluation_datasets/triples/train_dev_val/sft_test.jsonl \
    --answer-model dizza01/biomistral-7b-dare-triples-lora-merged \
    --answer-api-mode hf_endpoint \
    --answer-endpoint-url https://<your-biomistral-triples-endpoint-url> \
    --answer-endpoint-mode text_generation \
    --retrieval-mode default \
    --dense-top-k 5 \
    --answer-max-tokens 900 \
    --judge-max-tokens 700 \
    --external-judge-model meta-llama/Llama-3.1-70B-Instruct \
    --qwen-judge-model Qwen/Qwen2.5-72B-Instruct \
    --judge-retries 2 \
    --judge-retry-delay 0.5 \
    --run-name ab_biomistral7b_triples_endpoint_sfttest